 # Rapport de Projet - Application "Coach Sportif"
  Le Corre Tom, Karunakaran Rishikaran 

#### 1.1 Objectif du Projet

L'objectif principal de ce projet était de développer une application web (WebApp) interactive en Python. En utilisant la bibliothèque Streamlit, nous avons créé un "Coach Sportif" virtuel. L'application est conçue pour collecter les informations physiologiques et les objectifs d'un utilisateur, puis de générer des plans d'entraînement et de nutrition complets et personnalisés sur une période de quatre semaines.

#### 1.2 Architecture du Code

Pour garantir un code propre, modulable et facile à maintenir, nous avons adopté une architecture séparant la logique de l'affichage :

**interface2.py (Front-End)** : Ce fichier **gère exclusivement** l'interface utilisateur (UI) et les interactions. Il utilise Streamlit pour créer les formulaires, les boutons et afficher les résultats (tableaux, graphiques, PDF).

**calculs2.py (Back-End)** : Ce fichier est le **"cerveau"** de l'application. Il contient toutes les fonctions Python pour les calculs (IMC, calories), la logique de génération des plans (sport et nutrition) et la création des graphiques et des PDF.

#### 1.3 Technologies Utilisées
Pour réaliser ce projet, nous avons combiné plusieurs bibliothèques Python :

**Streamlit** : Pour la création de l'interface web.

**Pandas** : Pour la structuration des plans sur 4 semaines sous forme de DataFrame.

**Matplotlib & Plotly** : Pour la visualisation des données (graphique de prévision et camembert des macros).

**ReportLab** : Pour la génération dynamique des fichiers PDF exportables.

**Numpy & Random** : Pour les calculs et la sélection aléatoire des exercices/repas.

# 2/Saisie des Données Utilisateur (interface2.py)

La première étape de l'application est de **collecter les informations de l'utilisateur**. Nous avons choisi d'utiliser la barre latérale **(sidebar)** de Streamlit pour ne pas encombrer la page principale.

Nous avons utilisé des widgets spécifiques pour guider l'utilisateur et limiter les erreurs de saisie :

**st.number_input** pour l'âge, le poids et la taille, en définissant des valeurs minimales et maximales.

**st.radio** pour le sexe.

**st.selectbox** pour l'objectif et le niveau, offrant des choix prédéfinis.

Pour déclencher les calculs, nous utilisons un **st.button**. L'état de ce bouton est stocké dans **st.session_state['programme_genere']**, ce qui permet à l'application de **"mémoriser"** que l'utilisateur a demandé le plan et de garder le dashboard affiché même après une interaction.

In [ ]:
# 1) Saisie des informations
st.sidebar.header('Tes informations')
age = st.sidebar.number_input('Âge', min_value=10, max_value=99, value=25)
poids = st.sidebar.number_input('Poids (kg)', min_value=40, max_value=150, value=70)
taille = st.sidebar.number_input('Taille (cm)', min_value=140, max_value=210, value=170)
sexe = st.sidebar.radio('Sexe', ('Homme', 'Femme'), horizontal=True)
objectif = st.sidebar.selectbox('Objectif', ('Perte de poids', 'Prise de masse', 'Remise en forme'))
niveau = st.sidebar.selectbox('Niveau sportif', ('Debutant', 'Intermediaire', 'Avance'))

st.sidebar.write('---')

if st.sidebar.button('Générer mon programme'):
    st.session_state['programme_genere'] = True

# 3/Calculs de Base - IMC et Calories (calculs2.py)

Une fois le bouton "Générer" cliqué, **interface2.py appelle les premières fonctions de calculs2.py pour obtenir les métriques de base**.

##### 1. Calcul de l'IMC (calculer_imc)

Cette fonction calcule l'IMC et retourne non seulement la valeur numérique (arrondie à une décimale), mais aussi une interpretation textuelle (ex: "Poids normal", "Surpoids"), qui est plus parlante pour l'utilisateur.

##### 2. Calcul des Calories (calculer_calories)

C'est le calcul le plus important pour la nutrition. Nous avons utilisé la formule de Mifflin-St Jeor pour estimer le Métabolisme de Base (BMR). Ce BMR est ensuite ajusté selon deux critères :

Niveau d'activité : Un facteur (ex: 1.4 pour "Intermediaire") est appliqué au BMR pour obtenir la Dépense Énergétique Journalière (TDEE).

**Objectif** : La TDEE est ensuite multipliée par un modificateur (ex: * 0.85 pour une perte de poids, * 1.15 pour une prise de masse) pour déterminer l'apport calorique final conseillé.

In [ ]:
def calculer_calories(poids, taille_cm, age, sexe, niveau_sport, objectif):
    if sexe == 'Homme':
        bmr = (10 * poids) + (6.25 * taille_cm) - (5 * age) + 5
    else:
        bmr = (10 * poids) + (6.25 * taille_cm) - (5 * age) - 161

    # Doit correspondre aux valeurs envoyées par interface2.py
    facteur = {'Debutant': 1.2, 'Intermediaire': 1.4, 'Avance': 1.6}.get(niveau_sport, 1.3)
    tdee = bmr * facteur

    if objectif == 'Perte de poids':
        tdee *= 0.85
    elif objectif == 'Prise de masse':
        tdee *= 1.15

    return round(tdee)

# 4/Génération du Plan d'Entraînement (calculs2.py)

La fonction **generer_programme_sport** est le cœur de la partie entraînement. Elle génère un plan complet sur 4 semaines sous forme de DataFrame Pandas.

Logique de génération :

Base d'exercices : Un dictionnaire exercices stocke des listes d'exercices classés par type ('Haut du corps', 'Cardio', etc.).

**Paramètres de niveau** : Le niveau de l'utilisateur (Debutant, Intermediaire, Avance) détermine les variables jours_actifs, series, reps, et repos.

**Paramètres d'objectif** : L'objectif détermine la focus (une liste de types d'entraînement priorisés, ex: plus de 'Cardio' pour la perte de poids).

**Génération** : La fonction boucle sur 4 semaines et 7 jours.

Pour les j**ours_actifs**, elle sélectionne aléatoirement (random.sample) 2 exercices dans la catégorie focus appropriée.

Pour les jours de repos, elle assigne un "Repos actif" (Mobility/Stretch).

**Output** : La fonction retourne un DataFrame Pandas, que interface2.py affiche simplement avec st.dataframe.

In [ ]:
def generer_programme_sport(niveau, objectif):
    exercices = {
        'Haut du corps': [
            'Pompes', 'Tractions', 'Développé couché', # ...
        ],
        # ... autres catégories ...
    }

    if niveau == 'Debutant':
        jours_actifs = 3
        series = 2 # ...
    # ... autres niveaux ...

    if objectif == 'Perte de poids':
        focus = ['Full body', 'Cardio', 'Abdos', 'Mobility/Stretch', 'Bas du corps']
    # ... autres objectifs ...

    objectifs_hebdo = [
        'Semaine 1 : Stabiliser la technique et la régularité',
        # ... autres semaines ...
    ]

    lignes = []
    for semaine in range(1, 5):
        # ... (logique de boucle pour jours actifs et repos) ...
            categorie = focus[(semaine + j) % len(focus)]
            exos = ', '.join(random.sample(exercices[categorie], k=min(2, len(exercices[categorie]))))
            lignes.append({
                'Semaine': f'Semaine {semaine}',
                'Jour': f'Jour {jour}',
                'Type': categorie,
                # ... autres colonnes ...
            })
    return pd.DataFrame(lignes)

# 5/Génération du Plan Nutritionnel (calculs2.py)

Similaire au plan sportif, la fonction **generer_plan_nutrition** crée un plan de repas détaillé sur 4 semaines (28 jours) dans un DataFrame Pandas.

Logique de génération :

Base de données de repas : Nous avons créé un **dictionnaire** base très structuré. Il contient des suggestions de repas classées par **moment de la journée **(Petit-déjeuner, Déjeuner, etc.) et par **objectif**.

Portions incluses : Surtout, chaque suggestion est un tuple contenant le nom du repas ET les portions suggérées (ex: ('Poulet + légumes vapeur + quinoa', '120 g poulet, 200 g légumes, 60 g quinoa')).

Répartition calorique : Un dictionnaire repartition assigne un pourcentage des calories journalières à chaque repas (ex: 'Déjeuner': 0.35).

Génération : La fonction boucle sur 28 jours et 4 repas.

Elle utilise random.choice pour sélectionner une proposition de repas adaptée à l'objectif.

Elle calcule les calories estimées pour ce repas (calories * ratio).

Pour éviter la monotonie, elle **garde** en mémoire le **dernier_choix** et tente de ne pas proposer le même repas deux fois de suite.

**Output** : Elle retourne un DataFrame Pandas complet (28 jours * 4 repas = 112 lignes).

In [ ]:
def generer_plan_nutrition(objectif, calories):
    base = {
        'Petit-déjeuner': {
            'Perte de poids': [
                ('Flocons d\'avoine + yaourt + fruits rouges', '50 g avoine, 150 g yaourt, 100 g fruits'),
                ('Omelette + pain complet', '2 oeufs, 1 tranche pain complet'),
                # ...
            ],
            'Prise de masse': [
                ('Pancakes + beurre de cacahuète + banane', '2 pancakes, 1 càs beurre cacahuète, 1 banane'),
                # ...
            ], # ...
        }, # ...
    }
    repartition = {'Petit-déjeuner': 0.25, 'Déjeuner': 0.35, 'Collation': 0.15, 'Dîner': 0.25}
    lignes = []
    # ... (logique de boucle pour 4 semaines / 7 jours / 4 repas) ...
    return pd.DataFrame(lignes)

# 6/Visualisation des Macros (Plotly)

Pour compléter le plan nutritionnel, nous avons ajouté une visualisation des macronutriments (Protéines, Glucides, Lipides).

### Étape 1 : Logique (calculs2.py)

La fonction **get_macros_info** sert de base de données. En fonction de l'objectif de l'utilisateur, elle retourne deux dictionnaires :

macros : La répartition en pourcentage (ex: {'Protéines': 0.35, ...}).

aliments : Des listes d'aliments conseillés pour chaque macro.

### Étape 2 : Affichage (interface2.py)

Nous utilisons **Plotly Express (px.pie)** pour créer un graphique camembert interactif.

**st.plotly_chart(fig_pie)** est utilisé pour afficher le graphique dans Streamlit.

Nous affichons ensuite les listes d'aliments conseillés en utilisant st.markdown pour un formatage en gras.

In [ ]:
# 5) Macros + Aliments associés
st.header('4️⃣ Répartition des macronutriments')

macros, aliments = get_macros_info(objectif)
fig_pie = px.pie(
    names=list(macros.keys()),
    values=list(macros.values()),
    title=f'Répartition des nutriments pour ton objectif : {objectif}',
    color_discrete_sequence=px.colors.sequential.RdBu
)
st.plotly_chart(fig_pie, use_container_width=True)

st.subheader('Aliments conseillés pour chaque catégorie')
for macro, liste in aliments.items():
    st.markdown(f'**{macro} :** {", ".join(liste)}')

# 7/Visualisation de la Prévision (Matplotlib)

Pour motiver l'utilisateur, nous avons ajouté un graphique linéaire montrant son évolution de poids prévisionnelle sur les 4 semaines du plan.

### Étape 1 : Logique (calculs2.py)

La fonction **prevision_poids** utilise Matplotlib pour générer la figure :

Elle définit une base de variation de poids hebdomadaire selon l'objectif (ex: -0.45 kg pour "Perte de poids").

Elle définit des multiplicateurs (mults) basés sur le niveau d'activité (ex: 0.75 pour "Debutant").

Point clé : Elle trace les 3 courbes (Debutant, Intermediaire, Avance) sur le même graphique (ax.plot).

Elle met en surbrillance la courbe de l'utilisateur (linewidth=2.8) et laisse les autres en pointillés (linestyle='--') pour comparaison.

La fonction retourne la fig Matplotlib.

### Étape 2 : Affichage (interface2.py)

L'interface affiche simplement cette figure en utilisant st.pyplot(fig).

In [ ]:
def prevision_poids(poids, objectif, niveau_selectionne):
    semaines = np.arange(1, 5)
    # ... (logique de base et mults) ...
    fig, ax = plt.subplots(figsize=(7, 4))

    for niv, m in mults.items():
        p = poids
        courbe = []
        for _ in semaines:
            p += base * m
            courbe.append(round(p, 2))

        style = '-' if niv == niveau_selectionne else '--'
        width = 2.8 if niv == niveau_selectionne else 1.6
        # ...
        ax.plot(semaines, courbe, style,
                linewidth=width, alpha=alpha, label=niv, marker='o')
    
    # ... (mise en forme du graphique) ...
    return fig

# 8/Fonctionnalité Avancée - Export PDF (ReportLab)

La fonctionnalité la plus complexe de ce projet est l'exportation des plans en PDF. Nous avons utilisé la bibliothèque ReportLab.

#### Logique (calculs2.py)

Nous avons créé deux fonctions d'exportation : **exporter_programme_sport_pdf** et **exporter_plan_pdf**. Le processus est le suivant :

Buffer Mémoire : Nous créons un buffer en mémoire avec io.BytesIO().

Document : Nous initialisons un SimpleDocTemplate (de ReportLab) pointant vers ce buffer.

Contenu (Story) : Nous construisons le PDF en ajoutant des éléments à une liste story.

Nous ajoutons des Paragraph (pour les titres) et des Spacer (pour l'espacement).

Tableaux : Pour afficher les plans, nous filtrons le DataFrame Pandas semaine par semaine (ou jour par jour), le convertissons en liste, et créons un objet Table de ReportLab.

Style : Nous appliquons un TableStyle pour définir les couleurs de fond (BACKGROUND), la couleur du texte (TEXTCOLOR) et les grilles (GRID).

Pages : Nous ajoutons un PageBreak après chaque semaine pour que le PDF soit bien structuré.

Génération : pdf.build(story) compile le PDF dans le buffer.

Output : La fonction retourne les données binaires du PDF (buffer.getvalue()).

In [ ]:
def exporter_programme_sport_pdf(df_sport):
    buffer = io.BytesIO()
    pdf = SimpleDocTemplate(buffer, pagesize=A4)
    styles = getSampleStyleSheet()
    story = [Paragraph('Programme d\'entraînement – 4 semaines', styles['Title']), Spacer(1, 12)]

    for s in range(1, 5):
        story.append(Paragraph(f'Semaine {s}', styles['Heading2']))
        data = df_sport[df_sport['Semaine'] == f'Semaine {s}'][[
            'Jour', 'Type', 'Exercices', 'Séries', 'Répétitions', 'Repos', 'Durée (min)'
        ]]
        table_data = [list(data.columns)] + data.values.tolist()
        t = Table(table_data, colWidths=[50, 90, 190, 40, 60, 40, 60])
        t.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('GRID', (0, 0), (-1, -1), 0.25, colors.black)
        ]))
        story.append(t)
        if s < 4:
            story.append(PageBreak())

    pdf.build(story)
    pdf_data = buffer.getvalue()
    buffer.close()
    return pdf_data

# 9/Intégration de l'Export PDF (interface2.py)

Dans le fichier d'interface, nous utilisons **st.download_button** pour permettre à l'utilisateur de télécharger les PDF générés.

Cette fonction est très pratique :

Nous l'appelons juste après avoir affiché le DataFrame du plan correspondant.

Nous lui passons les données binaires (data=pdf_sport) que notre fonction ReportLab a générées.

Nous définissons le label (le texte du bouton), le file_name (nom du fichier téléchargé) et le mime (type de fichier).

In [ ]:
# Bouton de téléchargement PDF (Sport)
    pdf_sport = exporter_programme_sport_pdf(df_sport)
    st.download_button(
        label='Télécharger le programme sportif (PDF)',
        data=pdf_sport,
        file_name='programme_sportif.pdf',
        mime='application/pdf'
    )

    # ... (partie nutrition) ...

    # Bouton de téléchargement PDF (Nutrition)
    pdf_plan = exporter_plan_pdf(df_plan)
    st.download_button(
        label='Télécharger le plan nutritionnel (PDF)',
        data=pdf_plan,
        file_name='plan_nutritionnel.pdf',
        mime='application/pdf'
    )

# Conclusion

Ce projet a été une excellente opportunité de mettre en pratique nos compétences en Python dans un contexte d'application web concrète. La principale réussite a été l'intégration de multiples bibliothèques qui communiquent entre elles :

Streamlit gère l'interface et les interactions.

Pandas structure les données complexes (les plans sur 4 semaines).

Plotly et Matplotlib fournissent les visualisations de données.

ReportLab transforme les données de Pandas en rapports PDF professionnels.

La séparation de la logique (calculs2.py) et de l'affichage (interface2.py) a été une décision clé qui a rendu le projet gérable et a permis une collaboration efficace. L'application finale est fonctionnelle, répond à toutes les exigences initiales et offre une réelle valeur ajoutée à l'utilisateur grâce à ses plans détaillés et ses fonctionnalités d'export.